### Full timeline generation using Qwen3-32B

In [ ]:
import os
import re
from tqdm import tqdm
import json
import ast
import glob
import csv
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "/home/jainv/nlp_models/qwen3_32B/"
OUTPUT_PATH = "/home/jainv/dr_osb_lab/ChemoTask/dev_timelines_subtask2_7.csv"
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def extract_and_parse_json(text):
    """
    Scans `text` for the first JSON object or array and returns it as
    a Python object. Raises ValueError if none found or if parsing fails.
    """
    # look for the start of a JSON array
    match = re.search(r'([{\[])\s*', text)
    if not match:
        raise ValueError("No JSON object or array start found.")

    start_char = match.group(1)
    # choose matching closing char
    closing = '}' if start_char == '{' else ']'
    open_count = 0
    in_string = False
    escape = False

    for i in range(match.start(), len(text)):
        c = text[i]

        if in_string:
            if escape:
                escape = False
            elif c == '\\':
                escape = True
            elif c == '"':
                in_string = False
        else:
            if c == '"':
                in_string = True
            elif c == start_char:
                open_count += 1
            elif c == closing:
                open_count -= 1
                if open_count == 0:
                    json_str = text[match.start():i+1]
                    # try parsing
                    return json.loads(json_str)
                
    print("Incomplete JSON (could not find matching closing).")
    print(text)
    return None

In [ ]:
if os.path.exists(OUTPUT_PATH):
    pass
else:
    with open(OUTPUT_PATH, 'w') as f:
        out = csv.writer(f)
        out.writerow(["patient_num", "cancer_type", "timeline"])

sys_prompt2 = """You are helpful medical assistant, specialized in extracting chemotherapy timeline triplets from clinical notes.

When you receive two fields:
1. "detected_agents": an array of chemo‐agent names (strings) suggested by an external NER model.
2. "note_text": the full clinical note as a string.

Your task:
- **Primary**: Extract all chemotherapy agents mentioned in the note, whether or not they appear in "detected_agents".
- **Secondary**: Use "detected_agents" to guide your focus, but do **not** restrict yourself to them. If you find additional chemotherapy drug/regimen names and date pairs, include them too.
- For **each** extracted agent mention:
  1. Locate the exact time expression as it appears in the text and then **normalize** it in only one of these formats (YYYY-MM-DD OR YYYY-MM OR YYYY-wXX) where XX is the week number, using Principal Date as the reference. Always double check you have normalized all the dates
  2. Assign one of:
     - `begins-on`: when the note states the chemo **started** (e.g. “started on June 3, 2020”).
     - `ends-on`: when it states the chemo **finished** (e.g. “completed in September 2020”).
     - `contains-1`: when it indicates treatment **within** a period (e.g. “during January 2014”) without pinpointing a clear start or end.
- Always perform a check to ensure each triplet is valid for this format [chemo_agent, relation, normalized_timex3]
- Always **Output** **only** a valid JSON array of triplets `[chemo_agent, relation, normalized_timex3]`. No extra keys or commentary.

Example:
[
  ["taxol", "begins-on", "2020-06-03"],
  ["carboplatin","ends-on", "2020-09"],
  ["cisplatin", "contains-1", "2014-01"]
]"""


usr_prompt = """Here is the clinical note text. Please extract all chemotherapy timeline triplets as a JSON array of [chemo_agent, relation, normalized_timex3], using the built-in explanations of each relation type.
detected agents :
{detected_agents}

note text: 
{note_text}
"""

base_dir = '/home/jainv/dr_osb_lab/ChemoTask'
chemo_agents = pd.read_csv(f'{base_dir}/dev_entities_events_subtask2.csv')
completed = glob.glob("../dev_timelines_subtask2_*.csv")
lst = set()
for i in completed:
    df = pd.read_csv(i)
    lst.update(df['patient_num'].to_list())
chemo_agents = chemo_agents[(chemo_agents['events_list'].str.len()>2) & (~chemo_agents['patient_num'].isin(lst))]
    
for p_num, group in tqdm(chemo_agents.groupby('patient_num')):
    timelines = []
    for idx, row in tqdm(group.iterrows()):    
        raw = row['events_list']
        try:
            events_list = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            events_list = raw
        ents = set([i['entity'] for i in events_list])
        
        note_text = open(f"{row['note_path']}", 'r').read()
        messages = [
            {"role": "system", "content": sys_prompt2},
            {"role": "user", "content": usr_prompt.format(detected_agents=ents, note_text=note_text)},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=32768,
        )       

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

        try:
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0

        content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        json_content = extract_and_parse_json(content)

        if json_content:
            timelines.extend(json_content)
            
    print(timelines)
    with open(OUTPUT_PATH, 'a', newline='') as f:
        out = csv.writer(f)
        out.writerows(list(zip([p_num], [row["cancer_type"]], [timelines])))
    print("Completed: ", p_num)